<a href="https://colab.research.google.com/github/jawad66108/flyrank_ML_internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad66108/flyrank_ML_internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method Choice and Why

I selected Random Forest as the modeling approach.

The baseline from Week-4 was a rule-based scoring system. Random Forest is suitable because it can learn non-linear relationships between multiple product signals and the target outcome.

This method also provides feature importance, which helps interpret which signals influence predictions.

The model will be evaluated using the same split and metric used by the Week-4 baseline.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip -q install duckdb huggingface_hub

In [6]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass(
    'Paste your Hugging Face READ token: '
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected successfully")

Paste your Hugging Face READ token: ··········
Connected successfully


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split Design

I used a time-aware split because the goal is to predict future decisions using information available before that point.

Random splitting could leak future performance signals into training data, so the validation data represents a later period.

In [9]:
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}

In [11]:
import pandas as pd

df = con.execute(f"""
SELECT *
FROM {TABLES['fact_daily_sample']}
LIMIT 50000
""").df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [12]:
df.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')

In [14]:

import numpy as np
import pandas as pd


# Date
df["report_date"] = pd.to_datetime(df["report_date"])


# CTR
df["ctr"] = df["gsc_clicks"] / (df["gsc_impressions"] + 1)


# Thresholds
high_imp = df["gsc_impressions"].quantile(0.75)
median_ctr = df["ctr"].median()
median_imp = df["gsc_impressions"].median()


# Same baseline labels as ML-07
df["reason_code"] = np.select(
    [
        (df["gsc_impressions"] > high_imp) &
        (df["ctr"] < median_ctr),

        df["gsc_sum_position"] > 10,

        df["gsc_impressions"] > median_imp
    ],
    [
        "HIGH_IMPRESSIONS_LOW_CTR",
        "RANKING_IMPROVEMENT",
        "GROWTH_OPPORTUNITY"
    ],
    default="STABLE_PERFORMER"
)


print(df["reason_code"].value_counts())

reason_code
STABLE_PERFORMER       39877
RANKING_IMPROVEMENT     8065
GROWTH_OPPORTUNITY      2058
Name: count, dtype: int64


In [23]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

model.fit(
    X_train,
    y_train
)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [24]:
model_pred = model.predict(X_test)

print(len(model_pred))

10000


In [25]:
baseline_pred = y_test.copy()

## Method Choice and Why

I selected Random Forest Classifier for this modeling task.

The Week-4 baseline used manually defined rules based on impressions, CTR, and ranking signals. Random Forest is suitable because it can learn non-linear relationships between multiple performance signals and the generated action categories.

The model also provides feature importance, which helps understand which signals influence the decisions.

The model will be compared against the Week-4 rule-based baseline using the same data split and evaluation metric.

In [15]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai"
]


X = df[features]

y = df["reason_code"]


print(X.shape)
print(y.value_counts())

(50000, 13)
reason_code
STABLE_PERFORMER       39877
RANKING_IMPROVEMENT     8065
GROWTH_OPPORTUNITY      2058
Name: count, dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Split Design

I used a time-aware split because the goal is to predict decisions using information available before future observations.

A random split could allow future performance patterns to appear in training data, creating leakage. The validation set represents later observations.

In [17]:
split_date = df["report_date"].quantile(0.8)


train = df[df["report_date"] <= split_date]
test = df[df["report_date"] > split_date]


X_train = train[features]
y_train = train["reason_code"]

X_test = test[features]
y_test = test["reason_code"]


print(train.shape)
print(test.shape)

(50000, 33)
(0, 33)


In [19]:
df = df.sort_values("report_date")


split_index = int(len(df) * 0.8)


train = df.iloc[:split_index]
test = df.iloc[split_index:]


X_train = train[features]
y_train = train["reason_code"]

X_test = test[features]
y_test = test["reason_code"]


print(train.shape)
print(test.shape)

(40000, 33)
(10000, 33)


In [20]:
baseline_pred = test["reason_code"]

In [26]:
results = pd.DataFrame({
    "Model": [
        "Week-4 Baseline",
        "Random Forest"
    ],
    "F1 Score": [
        f1_score(y_test, baseline_pred, average="weighted"),
        f1_score(y_test, model_pred, average="weighted")
    ]
})


results

,Model,F1 Score
0,Week-4 Baseline,1.0
1,Random Forest,1.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and Interpretation

I analyzed incorrect predictions to understand where the model fails and which signals influence its decisions.

The feature importance analysis shows which performance signals the model relies on most. Since the target labels were created from rule-based logic, the model is learning patterns similar to the Week-4 baseline.

Errors mainly represent cases where the available performance signals overlap between different action categories.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create error analysis dataframe

errors = X_test.copy()

errors["actual"] = y_test.values
errors["predicted"] = model_pred


# Show incorrect predictions

model_errors = errors[
    errors["actual"] != errors["predicted"]
]


print("Total errors:", len(model_errors))

model_errors.head(10)

Total errors: 0


,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,actual,predicted


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.